# 04 · Momentum, mean-reversion & the cross-section

Two of the most studied effects in asset pricing, plus the factor-model decomposition of risk — all as sabia features on a **panel**:

- **Variance ratio** (`mean_reversion.var_ratio`) — Lo-MacKinlay (1988): VR > 1 means returns trend, VR < 1 means they mean-revert, VR = 1 is a random walk.
- **Cross-sectional momentum** (`cross_sectional.xs_rank_mom`) — Jegadeesh-Titman (1993): rank names by trailing 12-1 return; past winners keep beating past losers.
- **Market model** (`cross_sectional.beta`, `idio_vol`) — split each name's risk into market exposure and idiosyncratic vol.

The panel (`_simulate.momentum_panel`) is **ordered winner→loser**, with a permanent drift gradient (so momentum is real) and an AR(1) gradient (so the variance ratio spans trend↔reversion). Cross-sectional features need a complete panel and a declared `universe=`. Requires `pip install plotly nbformat jupyterlab`.

In [1]:
import sys, pathlib
HERE = pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

import sabia

import _simulate as sim

SCHEMA = sim.default_schema()

# A complete panel ordered winner -> loser: a permanent drift gradient (genuine cross-sectional
# winners/losers) and an AR(1) gradient (trend vs mean-reversion), sharing one market factor.
SYMBOLS = ("AAA", "BBB", "CCC", "DDD", "EEE", "FFF")
UNIVERSE = list(SYMBOLS)
panel = sim.momentum_panel(n=600, symbols=SYMBOLS)
panel.head(3)

timestamp,symbol,open,high,low,close,volume,vwap,dollar_volume,market_ret
"datetime[μs, UTC]",str,f64,f64,f64,f64,f64,f64,f64,f64
2021-01-01 00:00:00 UTC,"""AAA""",101.317289,101.933565,100.719489,101.39976,4.546187e6,101.350938,4.6098e8,-0.012514
2021-01-02 00:00:00 UTC,"""AAA""",99.747848,100.019962,99.145922,99.961356,3.828039e6,99.70908,3.8266e8,0.011674
2021-01-03 00:00:00 UTC,"""AAA""",99.453457,100.346382,99.275187,99.671173,4.585557e6,99.764247,4.5705e8,-0.007536


## 1 · Trend vs mean-reversion — the variance ratio
Computed per symbol. The ordering of the universe shows up directly: the high-drift/high-persistence names sit above the random-walk line, the mean-reverters below it.

In [2]:
# var_ratio(q): VR > 1 => positive return autocorrelation (trending); VR < 1 => mean-reverting
# (Lo-MacKinlay 1988). VR = 1 is the random-walk null.
vr = sabia.compute(panel, sabia.mean_reversion.var_ratio(q=5, window=126), schema=SCHEMA)
med = (
    panel.select("symbol")
    .hstack(vr)
    .group_by("symbol")
    .agg(pl.col("var_ratio_5_126").median().alias("vr"))
    .sort("symbol")
)
syms = med.get_column("symbol").to_list()
vals = med.get_column("vr").to_list()
colors = ["#2ca02c" if v >= 1 else "#d62728" for v in vals]
fig = go.Figure(go.Bar(x=syms, y=vals, marker_color=colors,
                       text=[f"{v:.2f}" for v in vals], textposition="outside"))
fig.add_hline(y=1.0, line=dict(color="#455a64", width=1.5, dash="dash"),
              annotation_text="random walk (VR = 1)")
fig.update_layout(height=400,
                  title="Variance ratio VR(5) by name — winners trend (VR&gt;1), losers mean-revert (VR&lt;1)",
                  yaxis_title="median VR(5, 126)", margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## 2 · Cross-sectional momentum
At each date, `xs_rank_mom` ranks every name's 12-1 momentum into a `[0,1]` percentile across the universe. Persistent green/red bands = rank persistence = momentum.

In [3]:
# Cross-sectional momentum rank: percentile of 12-1 momentum across the universe at each date.
xs = sabia.compute(
    panel,
    sabia.cross_sectional.xs_rank_mom(formation=126, skip=21),
    schema=SCHEMA, universe=UNIVERSE,
)
keyed = panel.select("timestamp", "symbol").hstack(xs).drop_nulls("xs_rank_mom_126")
wide = keyed.pivot(values="xs_rank_mom_126", index="symbol", on="timestamp").sort("symbol")
z = wide.drop("symbol").to_numpy()
cols = [c for c in wide.columns if c != "symbol"]
fig = go.Figure(go.Heatmap(z=z, x=cols, y=wide.get_column("symbol").to_list(),
                           colorscale="RdYlGn", zmin=0, zmax=1,
                           colorbar=dict(title="rank")))
fig.update_layout(height=360,
                  title="Cross-sectional momentum rank through time — winners stay green, losers stay red",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

A tradable read of the same signal: go long the top-third and short the bottom-third by **yesterday's** rank (strictly point-in-time), and accumulate. The drift gradient makes the spread grind upward.

In [4]:
# Winners-minus-losers: each day, long the top-third by *yesterday's* rank, short the bottom-third
# (strictly point-in-time), and accumulate the spread. Drift makes winners keep winning.
df = panel.select("timestamp", "symbol", "close").hstack(xs).with_columns(
    (pl.col("close").log() - pl.col("close").log().shift(1)).over("symbol").alias("ret"),
    pl.col("xs_rank_mom_126").shift(1).over("symbol").alias("rank_lag"),
).drop_nulls(["ret", "rank_lag"])
wml = df.group_by("timestamp").agg(
    (pl.col("ret").filter(pl.col("rank_lag") >= 0.6).mean()
     - pl.col("ret").filter(pl.col("rank_lag") <= 0.4).mean()).alias("wml")
).sort("timestamp").drop_nulls()
cum = (wml.get_column("wml").cum_sum() * 100).to_list()
fig = go.Figure(go.Scatter(x=wml.get_column("timestamp").to_list(), y=cum,
                           line=dict(color="#1f77b4", width=1.8), fill="tozeroy",
                           fillcolor="rgba(31,119,180,0.10)"))
fig.add_hline(y=0, line=dict(color="#455a64", width=1, dash="dash"))
fig.update_layout(height=380,
                  title="Cumulative winners-minus-losers return (top-third long, bottom-third short)",
                  yaxis_title="cumulative log-return (%)", margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## 3 · Market beta vs idiosyncratic vol
The factor-model features regress each name on the shared market factor. Total risk splits into a systematic part (beta) and a name-specific part (idiosyncratic vol).

In [5]:
# Factor-model features: regress each name's returns on the market factor (point-in-time).
fm = sabia.compute(
    panel,
    sabia.cross_sectional.beta(window=252),
    sabia.cross_sectional.idio_vol(window=252),
    schema=SCHEMA, universe=UNIVERSE,
)
last_ts = panel.get_column("timestamp").max()
snap = (
    panel.select("timestamp", "symbol")
    .hstack(fm)
    .filter(pl.col("timestamp") == last_ts)
    .sort("symbol")
)
fig = go.Figure(go.Scatter(
    x=snap.get_column("beta_252").to_list(),
    y=(snap.get_column("idio_vol_252") * np.sqrt(252) * 100).to_list(),
    mode="markers+text", text=snap.get_column("symbol").to_list(), textposition="top center",
    marker=dict(size=14, color=snap.get_column("beta_252").to_list(), colorscale="Viridis",
                showscale=False),
))
fig.update_layout(height=420,
                  title="Market beta vs idiosyncratic volatility (final date) — the two halves of total risk",
                  xaxis_title="beta(252)", yaxis_title="idiosyncratic vol (ann. %)",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## Takeaways

- `var_ratio` is a clean, single-number test of trend vs mean-reversion against the random-walk null.
- `xs_rank_mom` / `xs_z_mom` standardize a signal **across the universe** at each date — the basis of cross-sectional strategies. They require a complete panel and an explicit `universe=`, so completeness is checked, never inferred.
- `beta` and `idio_vol` are point-in-time market-model features — the inputs to risk models and the low-volatility anomaly.

Everything here is strictly trailing: the value at *t* uses only information at or before *t*.